# Sesión 2: Elementos Conformantes de una Base de Datos
## Fundamentos de Bases de Datos y Consultas Básicas

En esta sesión aprenderemos:
- Qué es una base de datos y sus tipos
- Usuarios y permisos (roles)
- Tablas y vistas
- Procedimientos almacenados y funciones

## 1. Setup Inicial

Crearemos una conexión SQLite en memoria para demostrar todos los conceptos.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json

# Crear conexión a base de datos SQLite en memoria
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("✅ Conexión SQLite establecida")

## 2. TABLAS: Estructura Física del Almacenamiento

Una tabla es la estructura fundamental para almacenar datos. Contiene:
- **Filas (registros):** cada fila es un registro
- **Columnas (campos):** cada columna tiene un tipo de dato
- **Clave primaria:** identifica únicamente cada fila
- **Restricciones:** validaciones (NOT NULL, CHECK, etc.)

### Crear tabla EMPLEADOS (Slide 16)

In [ ]:
# Crear tabla empleados con restricciones
cursor.execute('''
    CREATE TABLE empleados (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL,
        area TEXT,
        salario NUMERIC CHECK (salario > 0)
    )
''')

print("✅ Tabla 'empleados' creada exitosamente")
print("\nEstructura de tabla:")
print("- id: INTEGER (PRIMARY KEY)")
print("- nombre: TEXT (NOT NULL - obligatorio)")
print("- area: TEXT")
print("- salario: NUMERIC (CHECK - debe ser > 0)")

### Insertar datos en tabla

In [ ]:
# Insertar datos en tabla empleados
datos_empleados = [
    ('Ana', 'TI', 950000),
    ('Luis', 'Ventas', 800000),
    ('Marta', 'Marketing', 920000),
    ('Carlos', 'TI', 880000),
    ('Elena', 'Finanzas', 950000),
    ('Roberto', 'Ventas', 750000),
    ('Diana', 'Marketing', 870000),
    ('Fernando', 'Operaciones', 810000)
]

cursor.executemany(
    'INSERT INTO empleados (nombre, area, salario) VALUES (?, ?, ?)',
    datos_empleados
)

conn.commit()
print(f"✅ Se insertaron {cursor.rowcount} empleados")

# Ver datos
df = pd.read_sql_query('SELECT * FROM empleados', conn)
print("\nDatos insertados:")
print(df)

## 3. VISTAS: Estructura Lógica de Consulta (Slide 18)

Una vista es una consulta guardada que:
- NO almacena datos propios
- Actúa como filtro dinámico
- Restringe acceso (oculta columnas sensibles)
- Simplifica consultas complejas

### Crear vista para mostrar solo información pública

In [ ]:
# Crear vista que oculta salarios (información sensible)
cursor.execute('''
    CREATE VIEW vista_empleados_publicos AS
    SELECT 
        id,
        nombre, 
        area 
    FROM empleados
''')

print("✅ Vista 'vista_empleados_publicos' creada")
print("\nCaracterísticas:")
print("- Muestra: id, nombre, area")
print("- Oculta: salario (información sensible)")
print("- NO almacena datos propios")
print("- Es dinámica (refleja cambios en tabla empleados)")

# Ver datos desde la vista
df_vista = pd.read_sql_query('SELECT * FROM vista_empleados_publicos', conn)
print("\nDatos desde la vista:")
print(df_vista)

### Crear vista filtrada (empleados por área)

In [ ]:
# Crear vista solo para empleados de TI
cursor.execute('''
    CREATE VIEW empleados_ti AS
    SELECT 
        nombre,
        area,
        salario
    FROM empleados
    WHERE area = 'TI'
''')

print("✅ Vista 'empleados_ti' creada (solo área TI)")

df_ti = pd.read_sql_query('SELECT * FROM empleados_ti', conn)
print("\nEmpleados de TI:")
print(df_ti)

## 4. COMPARACIÓN: Tabla vs Vista

In [ ]:
comparacion = {
    'Criterio': [
        'Almacena datos',
        'Es editable',
        'Uso principal',
        'Rendimiento',
        'Seguridad'
    ],
    'Tabla': [
        'Sí - datos físicos',
        'Sí - INSERT, UPDATE, DELETE',
        'Almacenamiento',
        'Rápido (acceso directo)',
        'Requiere permisos para cada tabla'
    ],
    'Vista': [
        'No - solo consulta',
        'Parcialmente (según definición)',
        'Filtros, seguridad, reportes',
        'Depende de la consulta',
        'Filtra datos antes de mostrar'
    ]
}

df_comp = pd.DataFrame(comparacion)
print("TABLA vs VISTA:")
print(df_comp.to_string(index=False))

## 5. FUNCIONES: Cálculos y Transformaciones (Slide 22)

Una función es un conjunto de instrucciones que:
- Retorna un valor
- Puede usarse dentro de consultas SELECT
- Es útil para cálculos y validaciones

**Nota:** SQLite tiene soporte limitado para funciones personalizadas. Usaremos Python para simular.

### Definir función Python que simula función SQL

In [ ]:
# En PostgreSQL sería:
# CREATE FUNCTION calcular_bono(s NUMERIC) 
# RETURNS NUMERIC AS $$
# BEGIN
#     RETURN s * 0.10;
# END;
# $$ LANGUAGE plpgsql;

# Simulamos con Python:
def calcular_bono(salario, porcentaje=0.10):
    """Calcula bono como porcentaje del salario"""
    return salario * porcentaje

# Registrar función en SQLite
conn.create_function("calcular_bono", 1, calcular_bono)

print("✅ Función 'calcular_bono' creada")
print("\nUso:")
print("  SELECT nombre, salario, calcular_bono(salario) as bono FROM empleados")

# Usar función en consulta
df_bonus = pd.read_sql_query(
    'SELECT nombre, salario, calcular_bono(salario) as bono FROM empleados',
    conn
)
print("\nResultado:")
print(df_bonus)

### Otra función: Calcular aumento con criterio

In [ ]:
def calcular_aumento(salario):
    """Calcula aumento según rango de salario"""
    if salario < 800000:
        return salario * 0.10  # 10% para salarios bajos
    elif salario < 900000:
        return salario * 0.07  # 7% para salarios medios
    else:
        return salario * 0.05  # 5% para salarios altos

conn.create_function("calcular_aumento", 1, calcular_aumento)

print("✅ Función 'calcular_aumento' creada")
print("\nAumentos según criterio:")

df_aumento = pd.read_sql_query(
    '''SELECT 
        nombre, 
        area,
        salario,
        calcular_aumento(salario) as aumento,
        ROUND(salario + calcular_aumento(salario), 2) as nuevo_salario
    FROM empleados
    ORDER BY salario DESC''',
    conn
)
print(df_aumento)

## 6. PROCEDIMIENTOS ALMACENADOS (Slide 21)

Un procedimiento es un conjunto de instrucciones que:
- Ejecuta una tarea completa
- NO retorna necesariamente un valor
- Puede modificar datos (INSERT, UPDATE, DELETE)
- Se llama con CALL

**Nota:** SQLite no tiene soporte nativo. Usaremos Python para simular.

### Procedimiento para registrar nuevo empleado

In [ ]:
def registrar_empleado(nombre, area, salario):
    """Procedimiento: registra un nuevo empleado con validaciones"""
    try:
        # Validaciones
        if not nombre or len(nombre) < 2:
            raise ValueError("Nombre debe tener al menos 2 caracteres")
        
        if salario <= 0:
            raise ValueError("Salario debe ser mayor a 0")
        
        # Insertar
        cursor.execute(
            'INSERT INTO empleados (nombre, area, salario) VALUES (?, ?, ?)',
            (nombre, area, salario)
        )
        conn.commit()
        
        return f"✅ Empleado '{nombre}' registrado exitosamente"
    
    except ValueError as e:
        return f"❌ Error: {e}"
    except Exception as e:
        return f"❌ Error en la base de datos: {e}"

print("✅ Procedimiento 'registrar_empleado' definido")
print("\nEjemplo de uso:")

# Llamar procedimiento
resultado = registrar_empleado('Sofía', 'RRHH', 920000)
print(resultado)

# Verificar
df_nuevo = pd.read_sql_query(
    'SELECT * FROM empleados WHERE nombre = "Sofía"',
    conn
)
print("\nVerificación:")
print(df_nuevo)

### Procedimiento con validaciones más complejas

In [ ]:
def aplicar_bono_area(area, porcentaje):
    """Procedimiento: aplica bono a todos los empleados de un área"""
    try:
        if porcentaje < 0 or porcentaje > 0.50:
            raise ValueError("Porcentaje debe estar entre 0% y 50%")
        
        # Obtener empleados del área
        cursor.execute('SELECT id, salario FROM empleados WHERE area = ?', (area,))
        empleados = cursor.fetchall()
        
        if not empleados:
            return f"❌ No hay empleados en área '{area}'"
        
        # Actualizar salarios
        for emp_id, salario_actual in empleados:
            nuevo_salario = salario_actual * (1 + porcentaje)
            cursor.execute(
                'UPDATE empleados SET salario = ? WHERE id = ?',
                (nuevo_salario, emp_id)
            )
        
        conn.commit()
        return f"✅ Bono de {porcentaje*100}% aplicado a {len(empleados)} empleados de '{area}'"
    
    except Exception as e:
        return f"❌ Error: {e}"

print("✅ Procedimiento 'aplicar_bono_area' definido")
print("\nEjemplo: Aplicar 5% bono a área TI")

resultado = aplicar_bono_area('TI', 0.05)
print(resultado)

# Verificar
df_ti_bonificado = pd.read_sql_query(
    'SELECT nombre, area, salario FROM empleados WHERE area = "TI"',
    conn
)
print("\nEmpleados de TI con bono aplicado:")
print(df_ti_bonificado)

## 7. USUARIOS Y PERMISOS (Slide 12-14)

Los roles definen qué puede hacer cada usuario:
- **Administrador:** crear/eliminar objetos
- **Desarrollador:** crear tablas, modificar estructuras
- **Analista:** solo lectura (SELECT)
- **Auditor:** lectura sin alteración

### Simular roles y permisos

In [ ]:
# Definir roles con permisos
roles_permisos = {
    'administrador': {
        'SELECT': True,
        'INSERT': True,
        'UPDATE': True,
        'DELETE': True,
        'CREATE': True,
        'DROP': True
    },
    'desarrollador': {
        'SELECT': True,
        'INSERT': True,
        'UPDATE': True,
        'DELETE': False,
        'CREATE': True,
        'DROP': False
    },
    'analista': {
        'SELECT': True,
        'INSERT': False,
        'UPDATE': False,
        'DELETE': False,
        'CREATE': False,
        'DROP': False
    },
    'auditor': {
        'SELECT': True,
        'INSERT': False,
        'UPDATE': False,
        'DELETE': False,
        'CREATE': False,
        'DROP': False
    }
}

# Mostrar permisos por rol
df_roles = pd.DataFrame(roles_permisos).T
print("PERMISOS POR ROL:")
print(df_roles)
print("\nInterpretación:")
print("- True: permiso concedido")
print("- False: permiso denegado")

### Ejemplo: Otorgar permisos a un analista (en PostgreSQL sería)

In [ ]:
# En PostgreSQL real:
# GRANT SELECT ON empleados TO analista_rrhh;
# REVOKE INSERT, UPDATE, DELETE ON empleados FROM analista_rrhh;

print("En PostgreSQL, para otorgar permisos de lectura a un analista:")
print("""
-- Otorgar permiso de lectura a la tabla empleados
GRANT SELECT ON empleados TO analista_rrhh;

-- Asegurar que NO puede modificar datos
REVOKE INSERT, UPDATE, DELETE ON empleados FROM analista_rrhh;

-- Permite ver la vista (información filtrada)
GRANT SELECT ON vista_empleados_publicos TO analista_rrhh;
""")

print("\nResultado:")
print("- El analista puede: VER datos (SELECT)")
print("- El analista NO puede: Insertar, modificar, eliminar datos")
print("- Ventaja: Seguridad + acceso a información necesaria")

## 8. IMPACTO DE VISTAS EN SEGURIDAD

In [ ]:
print("ESCENARIO 1: Acceso directo a tabla (RIESGOSO)")
print("="*60)
print("Si un analista tiene acceso a la tabla 'empleados' sin restricciones:")
df_completa = pd.read_sql_query('SELECT * FROM empleados LIMIT 3', conn)
print(df_completa)
print("\n⚠️ PROBLEMA: Ve salarios (información sensible)")

print("\n" + "="*60)
print("\nESCENARIO 2: Acceso mediante vista (SEGURO)")
print("="*60)
print("Si accede mediante vista 'vista_empleados_publicos':")
df_vista_segura = pd.read_sql_query(
    'SELECT * FROM vista_empleados_publicos LIMIT 3',
    conn
)
print(df_vista_segura)
print("\n✅ VENTAJA: Solo ve información pública (nombre y área)")
print("✅ Los salarios están ocultos automáticamente")

## 9. ANÁLISIS COMPLETO: Estructura de Base de Datos de Empresa

In [ ]:
print("AUDITORÍA DE BASE DE DATOS - EMPRESA DE SERVICIOS")
print("="*70)

# 1. Identificar tablas
print("\n1. TABLAS EXISTENTES:")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tablas = cursor.fetchall()
for tabla in tablas:
    print(f"   - {tabla[0]}")

# 2. Identificar vistas
print("\n2. VISTAS EXISTENTES:")
cursor.execute("SELECT name FROM sqlite_master WHERE type='view'")
vistas = cursor.fetchall()
for vista in vistas:
    print(f"   - {vista[0]} (consulta guardada)")

# 3. Estadísticas
print("\n3. ESTADÍSTICAS:")
df_stats = pd.read_sql_query(
    '''SELECT 
        'Empleados totales' as metrica,
        COUNT(*) as valor
    FROM empleados''',
    conn
)
print(df_stats.to_string(index=False))

# 4. Análisis por área
print("\n4. DISTRIBUCIÓN POR ÁREA:")
df_areas = pd.read_sql_query(
    '''SELECT 
        area,
        COUNT(*) as cantidad,
        ROUND(AVG(salario), 2) as salario_promedio,
        MAX(salario) as salario_maximo
    FROM empleados
    GROUP BY area
    ORDER BY cantidad DESC''',
    conn
)
print(df_areas.to_string(index=False))

# 5. Seguridad
print("\n5. CONFIGURACIÓN DE SEGURIDAD (Simulada):")
print("   Rol 'Analista RRHH':")
print("   - Puede ver: vista_empleados_publicos (sin salarios)")
print("   - Permisos: SELECT (lectura)")
print("   - No puede: Insertar, modificar, eliminar datos")

print("\n   Rol 'Desarrollador':")
print("   - Puede ver: Todas las tablas")
print("   - Permisos: SELECT, INSERT, UPDATE, CREATE")
print("   - No puede: Eliminar tablas, DROP")

## 10. EJERCICIO PRÁCTICO: Implementar Solución

### Crear tabla adicional: DEPARTAMENTOS

In [ ]:
# Crear tabla departamentos
cursor.execute('''
    CREATE TABLE departamentos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL UNIQUE,
        presupuesto NUMERIC CHECK (presupuesto > 0),
        fecha_creacion TEXT
    )
''')

# Insertar departamentos
departamentos = [
    ('TI', 5000000, '2020-01-15'),
    ('Ventas', 3000000, '2020-01-15'),
    ('Marketing', 2500000, '2020-06-01'),
    ('Finanzas', 4000000, '2020-01-15'),
    ('RRHH', 1500000, '2021-03-10'),
    ('Operaciones', 2000000, '2020-02-01')
]

cursor.executemany(
    'INSERT INTO departamentos (nombre, presupuesto, fecha_creacion) VALUES (?, ?, ?)',
    departamentos
)
conn.commit()

print("✅ Tabla 'departamentos' creada")
df_depts = pd.read_sql_query('SELECT * FROM departamentos', conn)
print(df_depts)

### Crear vista de costo total por departamento

In [ ]:
# Crear vista que cruza empleados y departamentos
cursor.execute('''
    CREATE VIEW analisis_costo_departamentos AS
    SELECT 
        d.nombre as departamento,
        d.presupuesto,
        COUNT(e.id) as cantidad_empleados,
        ROUND(SUM(e.salario), 2) as costo_salarios,
        ROUND(d.presupuesto - SUM(e.salario), 2) as disponible
    FROM departamentos d
    LEFT JOIN empleados e ON d.nombre = e.area
    GROUP BY d.id, d.nombre, d.presupuesto
    ORDER BY costo_salarios DESC
''')

print("✅ Vista 'analisis_costo_departamentos' creada")
df_costo = pd.read_sql_query('SELECT * FROM analisis_costo_departamentos', conn)
print("\nAnálisis de costos por departamento:")
print(df_costo)

## 11. CASOS DE USO AVANZADOS

### Procedimiento: Generar reporte mensual

In [ ]:
def generar_reporte_mensual():
    """Procedimiento: genera reporte mensual de nómina"""
    try:
        # Calculamos totales
        df_reporte = pd.read_sql_query(
            '''SELECT 
                'REPORTE DE NÓMINA' as tipo,
                COUNT(*) as empleados,
                ROUND(SUM(salario), 2) as costo_total,
                ROUND(AVG(salario), 2) as salario_promedio,
                MAX(salario) as salario_maximo,
                MIN(salario) as salario_minimo
            FROM empleados''',
            conn
        )
        return df_reporte
    except Exception as e:
        print(f"Error: {e}")
        return None

print("✅ Procedimiento 'generar_reporte_mensual' definido\n")
df_reporte = generar_reporte_mensual()
print(df_reporte)

### Función: Clasificar empleado por nivel salarial

In [ ]:
def clasificar_nivel(salario):
    """Función: clasifica empleado por salario"""
    if salario < 800000:
        return 'Junior'
    elif salario < 900000:
        return 'Mid-level'
    else:
        return 'Senior'

conn.create_function("clasificar_nivel", 1, clasificar_nivel)

print("✅ Función 'clasificar_nivel' creada\n")

df_clasificado = pd.read_sql_query(
    '''SELECT 
        nombre,
        area,
        salario,
        clasificar_nivel(salario) as nivel
    FROM empleados
    ORDER BY salario DESC''',
    conn
)
print(df_clasificado)

## Conclusión

En esta sesión aprendimos los elementos conformantes de una base de datos:

✅ **TABLAS:** Almacenan datos físicamente con estructura definida  
✅ **VISTAS:** Filtran y restringen acceso sin almacenar datos propios  
✅ **FUNCIONES:** Realizan cálculos retornando valores  
✅ **PROCEDIMIENTOS:** Automatizan tareas complejas  
✅ **USUARIOS & PERMISOS:** Controlan quién accede a qué  

Estos elementos trabajan en conjunto para crear sistemas seguros, eficientes y auditables.